In [2]:
from datasets import load_dataset
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, ConfusionMatrixDisplay, confusion_matrix

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    PolynomialFeatures,
    FunctionTransformer,
)

from scipy.signal import welch

/home/daniela/Documentos/ia_code/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Definamos el "random_state" para que los resultados sean reproducibles:
random_state=42

# Análisis preliminar

## 1.0 Dataset loading:

In [4]:
# Carga el dataset:
data = load_dataset('YominE/Muscle_Fatigue_Cycling')
print(data)

# Conversion a dataframe de pandas para mejor visualización
df = data['train'].to_pandas()
print(df.head())

# Tamaño de dataset
print(f'Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas\n')

DatasetDict({
    train: Dataset({
        features: ['Time', 'Right Rectus femoris', 'Left Gluteus maximus', 'Left Gastrocnemius medialis', 'Left Semitendinosus', 'Left Biceps femoris caput longus', 'Right Vastus medialis', 'Right Tibialis anterior', 'Left Gastrocnemius lateralis', 'Target'],
        num_rows: 3002137
    })
})
    Time  Right Rectus femoris  Left Gluteus maximus  \
0  0.000             -0.000264             -0.000015   
1  0.001             -0.001002             -0.000045   
2  0.002             -0.002173             -0.000034   
3  0.003             -0.002676              0.000185   
4  0.004             -0.000844              0.000785   

   Left Gastrocnemius medialis  Left Semitendinosus  \
0                     0.000344             0.000108   
1                     0.001342             0.000429   
2                     0.002944             0.001133   
3                     0.003504             0.002319   
4                     0.000426             0.003950   

 

In [5]:
# Revisa tipos de datos y valores nulos:
print('=== Tipos de datos y nulos ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'nulos': df.isnull().sum(),
    '% nulos': (df.isnull().mean() * 100).round(2)
})
print(info)

=== Tipos de datos y nulos ===
                                    dtype  nulos  % nulos
Time                              float64      0      0.0
Right Rectus femoris              float64      0      0.0
Left Gluteus maximus              float64      0      0.0
Left Gastrocnemius medialis       float64      0      0.0
Left Semitendinosus               float64      0      0.0
Left Biceps femoris caput longus  float64      0      0.0
Right Vastus medialis             float64      0      0.0
Right Tibialis anterior           float64      0      0.0
Left Gastrocnemius lateralis      float64      0      0.0
Target                              int64      0      0.0


## 1.1.a Preprocessing data

In [6]:
print(f'Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas\n')
df.head()

#El tiempo se mide cada 0.001, es decir, un salto en el tiempo donde la frecuencia es 1000, las columnas son los musculos y cada dato en la columna hace referencia a las señales 
#eléctricas del musculo, los valores cercanos a 0 son poca actividad y los más cercanos a 1 es mayor actividad

# la frecuencia al ser 1000 significa que hay 1000 muestras por segundo

Dimensiones del dataset: 3,002,137 filas × 10 columnas



,Time,Right Rectus femoris,Left Gluteus maximus,Left Gastrocnemius medialis,Left Semitendinosus,Left Biceps femoris caput longus,Right Vastus medialis,Right Tibialis anterior,Left Gastrocnemius lateralis,Target
0,0.000,-0.000264,-0.000015,0.000344,0.000108,0.000182,0.000401,0.000267,-0.000236,0
1,0.001,-0.001002,-0.000045,0.001342,0.000429,0.000712,0.002234,0.001234,-0.001108,0
2,0.002,-0.002173,-0.000034,0.002944,0.001133,0.001692,0.007634,0.003457,-0.003277,0
3,0.003,-0.002676,0.000185,0.003504,0.002319,0.002820,0.017656,0.006587,-0.006940,0
4,0.004,-0.000844,0.000785,0.000426,0.003950,0.003729,0.028542,0.008889,-0.011310,0


In [7]:
# Reemplazar valores de target
df['Target'] = df['Target'].replace(2,1)
 
# Verificamos que target ahora solo contenga valor 0, 1
print(df['Target'].value_counts())

Target
0    2127600
1     874537
Name: count, dtype: int64


## 1.1.b Features clasification

In [8]:
TARGET = 'Target'

# Antes de clasificar features, separemos el target del dataset
X = df.drop(columns=[TARGET]) # Features
y = df[TARGET] # Goal

# Clasificación de features
continuous_num_cols = X.columns
print(continuous_num_cols)


Index(['Time', 'Right Rectus femoris', 'Left Gluteus maximus',
       'Left Gastrocnemius medialis', 'Left Semitendinosus',
       'Left Biceps femoris caput longus', 'Right Vastus medialis',
       'Right Tibialis anterior', 'Left Gastrocnemius lateralis'],
      dtype='object')


Explicación

Dado que todas las variables en df son numéricas, estas se pueden separar en binarias y continuas/discretas, pero la única binaria en nuestro df es la variable target, y dado que la clasificación de caracteristicas se hace sobre X, que es el df sin la feature target, el único tipo que nos queda son las numéricas continuas, que serían todas las features de X.

# Fetaures seleccionadas

## Justificación de la selección de características

Para la extracción de características a partir de las señales de electromiografía (EMG), se seleccionó un conjunto reducido pero representativo de variables que capturan diferentes dimensiones fisiológicas del comportamiento muscular durante el ejercicio. En particular, se eligieron las características **MAV (Mean Absolute Value), varianza, Zero Crossing (ZC)** y **frecuencia mediana**, ya que en conjunto permiten describir la señal tanto en el dominio del tiempo como en el de la frecuencia, evitando redundancias innecesarias y reduciendo el riesgo de sobreajuste.

- La **MAV (Mean Absolute Value)** mide el valor promedio absoluto de la señal, proporcionando una estimación directa del nivel de activación muscular. Valores altos de MAV se asocian con un mayor reclutamiento de unidades motoras, lo cual refleja el esfuerzo realizado por el músculo. Esta característica es ampliamente utilizada en el análisis de EMG debido a su simplicidad y estabilidad frente al signo de la señal.

- La **varianza** cuantifica la dispersión de los valores de la señal respecto a su media, permitiendo caracterizar la variabilidad o irregularidad del comportamiento muscular. Esta medida complementa a la MAV, ya que dos señales pueden tener niveles de activación similares pero diferir en su estructura interna. En el contexto de fatiga, una varianza alta puede indicar una señal más irregular o con cambios bruscos en la activación, lo cual puede estar asociado a compensaciones musculares o a una activación menos eficiente. Por otro lado, una varianza baja puede reflejar una señal más uniforme o “plana”, lo cual también puede aparecer en estados de fatiga avanzada donde el músculo pierde capacidad de respuesta. En este sentido, la varianza no indica fatiga de forma directa, pero sí aporta información sobre cambios en el comportamiento de la señal que ayudan al modelo a distinguir entre estados.

- El **Zero Crossing (ZC)** corresponde al número de veces que la señal cambia de signo dentro de una ventana. Más allá de ser un simple conteo, esta característica permite estimar qué tan rápido oscila la señal en el tiempo. Una señal que cambia de signo muchas veces en un intervalo corto implica que está oscilando rápidamente (es decir, tiene componentes de alta frecuencia), mientras que una señal con pocos cruces por cero cambia más lentamente (menor contenido de altas frecuencias). Por esta razón, el ZC actúa como un indicador indirecto del contenido en frecuencia sin necesidad de transformar la señal al dominio frecuencial. En el contexto de fatiga muscular, una disminución en el número de cruces por cero suele asociarse a una señal más lenta, lo cual es consistente con la reducción de componentes de alta frecuencia observada en músculos fatigados.

- La **frecuencia mediana** se define como la frecuencia que divide el espectro de potencia en dos mitades de igual energía. Esta característica es particularmente relevante en el análisis de fatiga muscular, ya que existe evidencia de que, a medida que el músculo se fatiga, el contenido espectral de la señal EMG se desplaza hacia frecuencias más bajas. Por tanto, la frecuencia mediana constituye un indicador robusto y fisiológicamente significativo del estado de fatiga.


## Exclusión de otras características

Se decidió no incluir ciertas características comúnmente utilizadas con el fin de evitar redundancia y reducir la dimensionalidad del problema.

En particular, el **RMS (Root Mean Square)** no fue considerado, ya que proporciona información muy similar a la MAV. Ambas métricas están altamente correlacionadas al medir la amplitud de la señal, por lo que incluir ambas no aportaría información adicional relevante y podría incrementar el riesgo de sobreajuste.

De manera similar, la **potencia espectral total** fue descartada debido a que su información se encuentra en gran medida representada por medidas como la varianza y la MAV, las cuales también están relacionadas con la energía de la señal.

Por otra parte, la **frecuencia media** no fue seleccionada, ya que, aunque describe el centro de masa del espectro, es más sensible a valores extremos y ruido en comparación con la frecuencia mediana. En el contexto del análisis de fatiga muscular, la frecuencia mediana resulta más robusta y confiable.


## Conclusión

El conjunto de características seleccionado permite capturar de manera equilibrada aspectos clave de la señal EMG: nivel de activación (MAV), variabilidad (varianza), dinámica temporal (ZC) y comportamiento espectral (frecuencia mediana). Esta selección busca maximizar la capacidad del modelo para discriminar entre estados de fatiga y no fatiga, manteniendo al mismo tiempo una baja redundancia y reduciendo el riesgo de sobreajuste.


In [ ]:
def frecuencia_mediana(ventana, fs):
    """Función para obtener la frecuencia_mediana"""
    freqs, psd = welch(ventana, fs=fs)
    potencia_acumulada = np.cumsum(psd)
    frecuencia_media_idx = np.searchsorted(potencia_acumulada, potencia_acumulada[-1] / 2)
    
    return freqs[frecuencia_media_idx]

In [10]:
def  extract_features(ventana, fs=1000):
    """Extrae características de tiempo y frecuencia de una ventana 1D."""
    # --- Dominio del tiempo ---
    mav     = np.mean(np.abs(ventana))
    varianza = np.var(ventana)
    zcr     = np.sum(np.diff(np.sign(ventana)) != 0)

    # --- Dominio de la frecuencia ---
    freqs, psd = welch(ventana, fs=fs)
    pot_acum   = np.cumsum(psd)
    idx_median = np.searchsorted(pot_acum, pot_acum[-1] / 2)
    frec_mediana = freqs[idx_median]

    return [mav, varianza, zcr, frec_mediana]

In [11]:
print(len(df))

3002137


In [ ]:
""" Creamos nuevo dataframe agrupando 1000 filas del viejo df, en 1 sola, esto lo hacemos guardando información
especifica (features como mav, etc) de esas filas, en una sola, que contenga las features de todos los musculos """
# las ventanas nos permiten tener en un espacio de tiempo de 1 segundo agrupar 1000 datos de todos los musculos 

fs = 1000
windows_size = fs  # como habiamos dicho entonces 1 segundo = 1000  datos 
channels = [col for col in df.columns if col not in ["Time", "Target"]] # channels son las columnas del viejo datatset (los musculos)
rows = [] # contendrá todas las row del dataframe nuevo, 1 row serían las 32 features en 1s 

for i in range(0, len(df) - windows_size, windows_size): #creamos el ciclo que recorre todas las señales (datos) y lo corta en las ventanas que queremos 
    # contiene todos los musculos en 1s de dataset viejo
    window_df = df.iloc[i:i+windows_size]  # contiene una ventana del dataset viejo, treyendo todos los musculos en ese i.

    row = {} # contendrá fila de los 8 musculos, con sus 4 features
    for channel in channels:  # recorremos los musculos
        signals = window_df[channel].values  # tomamos las 1000 señales de una columna en especifico
        features = extract_features(signals, fs)

        features_names = ['mav', 'var', 'zcr', 'f_mediana']

        for name,value in zip(features_names, features):
            row[f'{channel}_{name}'] = value

    #se le caclula las 4 features a los 8 musculos, y se guardan -> al final cada ventana es una fila y cada musculo pone varias columnas de cada feature
    # como en cada ventana se arupan 1000 filas entonces se debe definir un solo target para la ventana, este se saca con la moda 
    row['target'] = window_df['Target'].mode()[0]

    rows.append(row)  #cada fiila de feature es una ventana con las 4 caracteristicas de cada uno de los 8 musculos 

# nuevo dataset
df_with_features = pd.DataFrame(rows)

#print("nuevo dataframe")
print(df_with_features.head())  #se crea el dataset pero ya estructurado con la info dada en las caracteristicas/features




   Right Rectus femoris_mav  Right Rectus femoris_var  \
0                  0.008125                  0.000137   
1                  0.010118                  0.000197   
2                  0.010151                  0.000220   
3                  0.009894                  0.000191   
4                  0.009676                  0.000177   

   Right Rectus femoris_zcr  Right Rectus femoris_f_mediana  \
0                       123                        50.78125   
1                       111                        54.68750   
2                       121                        46.87500   
3                       121                        46.87500   
4                       122                        46.87500   

   Left Gluteus maximus_mav  Left Gluteus maximus_var  \
0                  0.003260                  0.000016   
1                  0.003398                  0.000018   
2                  0.003354                  0.000018   
3                  0.003532                  0.000

In [14]:
print(len(df_with_features))

3002
